# 01 — Extract per-frame DINOv2 embeddings (Kaggle GPU)

Output: `features/dinov2b_train.npy` (3000,30,1536) and `dinov2b_test.npy` (1750,30,1536) in float16, plus id CSVs.
Each frame vector = [CLS token | mean of patch tokens] from `facebook/dinov2-base`.

**Settings (right panel):** Accelerator = GPU T4 x2 (we use one GPU), Internet = On (downloads weights), attach the competition `cooked-or-not`.
Runtime estimate: ~142k frames, a few minutes to ~15 min on a T4.

In [ ]:
import os, glob, time
import numpy as np, pandas as pd, torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoModel

dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev, torch.cuda.get_device_name(0) if dev == "cuda" else "(NO GPU - enable it in Settings > Accelerator)")

In [ ]:
# --- locate the data root (works on Kaggle and locally) ---
hits = glob.glob("/kaggle/input/**/train_labels.csv", recursive=True) + glob.glob("../../data/train_labels.csv")
assert hits, "train_labels.csv not found - attach the dataset / check the path"
ROOT = os.path.dirname(hits[0])
assert os.path.isdir(f"{ROOT}/train") and os.path.isdir(f"{ROOT}/test"), f"train/ or test/ missing under {ROOT}"
SMOKE = int(os.environ.get("SMOKE", 0)) or None   # local smoke test: SMOKE=50 uses only 50 clips per split
OUT = "/kaggle/working/features" if os.path.exists("/kaggle/working") else ("../../artifacts/features_smoke" if SMOKE else "../../artifacts/features")
os.makedirs(OUT, exist_ok=True)

lab = pd.read_csv(f"{ROOT}/train_labels.csv")
sub = pd.read_csv(f"{ROOT}/sample_submission.csv")
train_ids, test_ids = lab["clip_id"].tolist()[:SMOKE], sub["clip_id"].tolist()[:SMOKE]
print(ROOT, len(train_ids), len(test_ids))

In [ ]:
MODEL = "facebook/dinov2-base"      # no internet? attach a DINOv2 Kaggle Model and point MODEL to its path
model = AutoModel.from_pretrained(MODEL).to(dev).eval().half() if dev == "cuda" else AutoModel.from_pretrained(MODEL).eval()
mean = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
H, W = 224, 392   # DINOv2 patch size 14: 224 = 16*14, 392 = 28*14 (original frames are 384x224)

In [ ]:
class ClipDS(torch.utils.data.Dataset):
    def __init__(self, split, ids): self.split, self.ids = split, ids
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        d = f"{ROOT}/{self.split}/{self.ids[i]}"
        fr = [np.asarray(Image.open(f"{d}/frame_{j:03d}.jpg").convert("RGB").resize((W, H), Image.BICUBIC)) for j in range(30)]
        return torch.from_numpy(np.stack(fr)).permute(0, 3, 1, 2)   # uint8 (30,3,H,W)

@torch.no_grad()
def extract(split, ids, bs=4):
    dl = torch.utils.data.DataLoader(ClipDS(split, ids), batch_size=bs, num_workers=0 if os.name == "nt" else 2, shuffle=False)
    out = []
    for x in tqdm(dl, desc=split):
        b = x.shape[0]
        x = x.flatten(0, 1).to(dev).float().div(255)
        x = ((x - mean) / std)
        if dev == "cuda": x = x.half()
        h = model(pixel_values=x).last_hidden_state              # (b*30, 1+patches, 768)
        f = torch.cat([h[:, 0], h[:, 1:].mean(1)], dim=-1)        # (b*30, 1536)
        out.append(f.view(b, 30, -1).float().cpu().numpy().astype(np.float16))
    return np.concatenate(out)

In [ ]:
t0 = time.time()
Xtr = extract("train", train_ids); np.save(f"{OUT}/dinov2b_train.npy", Xtr)
Xte = extract("test", test_ids);   np.save(f"{OUT}/dinov2b_test.npy", Xte)
pd.Series(train_ids, name="clip_id").to_csv(f"{OUT}/train_ids.csv", index=False)
pd.Series(test_ids, name="clip_id").to_csv(f"{OUT}/test_ids.csv", index=False)
print(Xtr.shape, Xte.shape, f"{time.time()-t0:.0f}s", "NaNs:", np.isnan(Xtr).sum(), np.isnan(Xte).sum())

## Next (local)
Download: `kaggle kernels output <user>/cooked-or-not-01-extract-features -p artifacts/features`
Then locally: `pool_features()` from `src/common.py` -> logistic regression with `make_folds()` -> OOF AUC.